# Large frozen Omni zero-shot pilot

Qwen2.5-Omni-7B: 10.73B original checkpoint parameters, 8.93B in the
audio/video/text understanding model with speech generation disabled.
Frozen 4-bit language weights; FP16 audio/vision encoders; no fine-tuning.
The fixed prompt contains task definitions and unlabelled observations, not
labelled examples. Inference reads no reference labels. Score all 24 fixed,
uniformly spaced moments afterward and compare the same moments for each baseline.
This sparse classification pilot has no meaningful full-window spellF1.

Version 2 is a fixed follow-up after the original interface produced 25% accuracy.
It asks separately for play state and the controlling club, and compares explicit
tracking descriptions with audio/video alone. Both modes use exactly the original
24 moments; both are reported alongside version 1. No examples or weights are
trained. This prompt revision makes the results exploratory rather than an
independent validation of a prespecified prompt.


In [ ]:
from pathlib import Path
import hashlib
import json
import os
import shutil
import subprocess
import sys
import torch

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working/omni_zero_shot")
WORK.mkdir(parents=True, exist_ok=True)
assert torch.cuda.is_available()
print("GPU:", torch.cuda.get_device_name(0), "count", torch.cuda.device_count(), flush=True)
print("Free /tmp GiB:", shutil.disk_usage("/tmp").free / 1024**3, flush=True)
assert shutil.disk_usage("/tmp").free > 22 * 1024**3, "Need room for four original weight shards."
def sha(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()
roots = [p.parent for p in INPUT.rglob("requirements-foundation.txt")
         if (p.parent / "foundation_possession/pilot.py").is_file()]
assert len(roots) == 1
CODE = roots[0]
configs = list(INPUT.rglob("foundation_pilot_input.json"))
assert len(configs) == 1
LABELS = configs[0].parent
features = [p for p in INPUT.rglob("features.npz")
            if sha(p) == "dc8c85c46bb381eaa8b1bb6deeb004f458165237bb00b7e89ba4fc4a756f805a"]
assert len(features) == 1
FEATURES = features[0]
videos = list(INPUT.rglob("BAR-ATH_tactical.mp4"))
assert len(videos) == 1
VIDEO = videos[0]
CONTROLS = {}
for arm, expected in [("learned_audio", "4c6cdd0f389838c9e1eaf5ea5342c485d6ae6528b309b3528380236a7ae0ddc6"),
                      ("tracking", "8cb9edf9db0a8a4e8bce068f92cd3839f4151f68a659100506ee578ec3850f5a")]:
    paths = [p for p in INPUT.rglob("predictions.csv")
             if p.parent.name == arm and p.parent.parent.name == "comparison"]
    assert len(paths) == 1
    assert sha(paths[0]) == expected
    CONTROLS[arm] = paths[0]
cpu = [p for p in INPUT.rglob("learned.csv")
       if p.parent.name == "run" and "possession_run02" in p.parts]
assert len(cpu) == 1
CONTROLS["cpu"] = cpu[0]
print("Source:", CODE, "Video:", VIDEO, flush=True)


In [ ]:
DEPS = Path("/tmp/barca_omni_python")
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--disable-pip-version-check",
                "--target", str(DEPS), "-r", str(CODE / "requirements-learning.txt")], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--disable-pip-version-check",
                "--target", str(DEPS), "--no-deps", "-r", str(CODE / "requirements-foundation.txt")], check=True)
env = dict(os.environ, PYTHONPATH=str(DEPS) + os.pathsep + str(CODE),
           PYTHONDONTWRITEBYTECODE="1", HF_HOME="/tmp/barca_omni_hf",
           HF_HUB_DISABLE_XET="1", HF_HUB_DISABLE_TELEMETRY="1",
           HF_HUB_DOWNLOAD_TIMEOUT="60", HF_HUB_ETAG_TIMEOUT="30",
           HF_HUB_DISABLE_PROGRESS_BARS="1", CUBLAS_WORKSPACE_CONFIG=":4096:8")
def run(*args):
    subprocess.run([sys.executable, *map(str, args)], cwd=CODE, env=env, check=True)
run("-c", """
import torch,transformers,bitsandbytes as bnb
print('Foundation runtime:',transformers.__version__,bnb.__version__,torch.__version__,flush=True)
m=bnb.nn.Linear4bit(32,32,compute_dtype=torch.float16,quant_type='nf4').cuda()
with torch.inference_mode():y=m(torch.ones(1,32,device='cuda',dtype=torch.float16))
assert torch.isfinite(y).all()
print('CUDA NF4 backend works before large downloads.',flush=True)
""")
run("-m", "unittest", "discover", "-s", "tests", "-v")


Freeze the case plan and prompt before downloading large weights. Download only
the four original shards containing understanding-model tensors; speech-only
weights are omitted from the index. No tensor values are trained or changed.
All private video/audio processing occurs inside this notebook. Only model files
are downloaded from Hugging Face; private match media is not uploaded there.


In [ ]:
RUN = WORK / "pilot"
run("-m", "foundation_possession.pilot", "infer",
    "--features", FEATURES, "--video", VIDEO, "--out", RUN,
    "--first", "24958", "--stop", "29036", "--count", "24",
    "--interface", "dual",
    "--model-cache", "/tmp/barca_omni_model")


Inference is complete before this scoring command opens reference labels.
Invalid answers and X abstentions count as misses. Sparse accuracy/macro F1
are compared on the exact same 24 moments for every method. The draft reference
and two sampled D moments limit what this pilot can establish. No acoustic
ground truth is available for scoring whistle descriptions.


In [ ]:
for mode in ("semantic_tracking", "semantic_av_only"):
    destination = RUN / mode
    run("-m", "foundation_possession.pilot", "score",
        "--run", destination, "--truth", LABELS / "validation.csv",
        "--compare", CONTROLS["learned_audio"], CONTROLS["tracking"], CONTROLS["cpu"],
        CODE / "evaluation/results/possession_v6.csv")
    report = json.loads((destination / "report.json").read_text())
    for name, score in report["scores"].items():
        print(mode, name, "sparse accuracy", round(score["accuracy"],4),
              "macroF1", round(score["macro_f1"],4), "D_F1", round(score["dead_f1"],4),
              "coverage", score["coverage"], flush=True)
print("ZERO-SHOT PILOT COMPLETE", flush=True)
